# BRSR Data Preprocessing Pipeline
Reads all `.xlsx` files from the `Excel Files` folder and produces a `Master Sheet.xlsx` with four columns:
`company_name`, `year`, `element_name`, `fact_value`

In [1]:
import pandas as pd
import os
import re
from tqdm import tqdm

In [2]:
# ---------------------------------------------------------------------------
# TARGET ELEMENTS  (original CamelCase — do NOT lowercase these)
# ---------------------------------------------------------------------------
RELEVANT_ELEMENTS = {
    # General / Cross-Cutting
    "StatementByDirectorResponsibleForTheBusinessResponsibilityReportHighlightingESGRelatedChallengesTargetsAndAchievementsExplanatoryTextBlock",
    "DetailsOfTheHighestAuthorityResponsibleForImplementationAndOversightOfTheBusinessResponsibilityPolicyExplanatoryTextBlock",
    "DetailsOfSpecifiedCommitteeOfTheBoardOrDirectorResponsibleForDecisionMakingOnSustainabilityRelatedIssuesExplanatoryTextBlock",
    "NameOfTheNationalAndInternationalCodesOrCertificationsOrLabelsOrStandardsAdoptedByYourEntityAndMappedToEachPrincipleExplanatoryTextBlock",
    "SpecificCommitmentsGoalsAndTargetsSetByTheEntityWithDefinedTimelinesExplanatoryTextBlock",
    "PerformanceOfTheEntityAgainstTheSpecificCommitmentsGoalsAndTargetsAlongWithReasonsInCaseTheSameAreNotMetExplanatoryTextBlock",
    "MaterialIssueIdentified",
    "RationaleForIdentifyingTheRiskOpportunity",
    "InCaseOfRiskApproachToAdaptOrMitigate",
    "TopicsOrPrinciplesCoveredUnderTheTrainingAndItsImpact",

    # P1 — Ethics
    "AntiCorruptionOrAntiBriberyPolicyExplanatoryTextBlock",
    "DetailsOfTheEntityHaveProcessesInPlaceToAvoidOrManageConflictOfInterestsInvolvingMembersOfTheBoardExplanatoryTextBlock",
    "DetailsOfAnyCorrectiveActionTakenOrUnderwayOnIssuesRelatedToFinesOrPenaltiesOrActionTakenByRegulatorsOrLawEnforcementAgenciesOrJudicialInstitutionsOnCasesOfCorruptionAndConflictsOfInterestExplanatoryTextBlock",
    "BriefOfTheMonetaryCaseForPenaltyOrFineExplanatoryTextBlock",
    "BriefOfTheMonetaryCaseForSettlementExplanatoryTextBlock",
    "BriefOfTheMonetaryCaseForCompoundingFeeExplanatoryTextBlock",
    "BriefOfTheMonetaryCaseForImprisonmentExplanatoryTextBlock",
    "BriefOfTheMonetaryCaseForPunishmentExplanatoryTextBlock",

    # P2 — Sustainable Products
    "DescribeTheProcessesInPlaceToSafelyReclaimYourProductsForReusingRecyclingAndDisposingAtTheEndOfLifeForPlasticsIncludingPackagingExplanatoryTextBlock",
    "DescribeTheProcessesInPlaceToSafelyReclaimYourProductsForReusingRecyclingAndDisposingAtTheEndOfLifeForEWasteExplanatoryTextBlock",
    "DescribeTheProcessesInPlaceToSafelyReclaimYourProductsForReusingRecyclingAndDisposingAtTheEndOfLifeForHazardousWasteExplanatoryTextBlock",
    "DescribeTheProcessesInPlaceToSafelyReclaimYourProductsForReusingRecyclingAndDisposingAtTheEndOfLifeForOtherWasteExplanatoryTextBlock",
    "DetailsOfImprovementsInEnvironmentalAndSocialImpactsDueToCapex",
    "DetailsOfImprovementsInEnvironmentalAndSocialImpactsDueToRAndD",

    # P3 — Employee Wellbeing
    "DetailsOfMechanismAvailableToReceiveAndRedressGrievancesForPermanentWorkersExplanatoryTextBlock",
    "DetailsOfMechanismAvailableToReceiveAndRedressGrievancesForOtherThanPermanentWorkersExplanatoryTextBlock",
    "DetailsOfMechanismAvailableToReceiveAndRedressGrievancesForPermanentEmployeesExplanatoryTextBlock",
    "DetailsOfMechanismAvailableToReceiveAndRedressGrievancesForOtherThanPermanentEmployeesExplanatoryTextBlock",
    "DetailsOfOccupationalHealthAndSafetyManagementSystemExplanatoryTextBlock",
    "DesclosureOfTheProcessesUsedToIdentifyWorkRelatedHazardsAndAssessRisksOnARoutineAndNonRoutineBasisByTheEntityExplanatoryTextBlock",
    "DescribeTheMeasuresTakenByTheEntityToEnsureASafeAndHealthyWorkPlaceExplanatoryTextBlock",
    "DetailsOfAnyCorrectiveActionTakenOrUnderwayToAddressSafetyRelatedIncidentsOfYourPlantsAndOfficesThatWereAssessedExplanatoryTextBlock",

    # P4 — Stakeholder Engagement
    "DescribeTheProcessesForIdentifyingKeyStakeholderGroupsOfTheEntityExplanatoryTextBlock",
    "ProvideTheProcessesForConsultationBetweenStakeholdersAndTheBoardOnEconomicEnvironmentalAndSocialTopicsOrIfConsultationIsDelegatedHowIsFeedbackFromSuchConsultationsProvidedToTheBoardExplanatoryTextBlock",
    "DetailsOfInstancesAsToHowTheInputsReceivedFromStakeholdersOnTheseTopicsWereIncorporatedIntoPoliciesAndActivitiesOfTheEntityExplanatoryTextBlock",
    "DetailsOfInstancesOfEngagementWithAndActionsTakenToAddressTheConcernsOfVulnerableMarginalizedStakeholderGroupsExplanatoryTextBlock",
    "PurposeAndScopeOfEngagementIncludingKeyTopicsAndConcernsRaisedDuringSuchEngagement",

    # P5 — Human Rights
    "DescribeTheInternalMechanismsInPlaceToRedressGrievancesRelatedToHumanRightsIssuesExplanatoryTextBlock",
    "MechanismsToPreventAdverseConsequencesToTheComplainantInDiscriminationAndHarassmentCasesExplanatoryTextBlock",
    "DetailsOfAnyCorrectiveActionsTakenOrUnderwayToAddressSignificantRisksOrConcernsArisingFromTheAssessmentsOfPlantAndOfficeExplanatoryTextBlock",
    "DetailsOfABusinessProcessBeingModifiedOrIntroducedAsAResultOfAddressingHumanRightsGrievancesOrComplaintsExplanatoryTextBlock",

    # P6 — Environment
    "DetailsOfCoverageAndImplementationIfForZeroLiquidDischargeExplanatoryTextBlock",
    "DetailsOfProjectRelatedToReducingGreenHouseGasEmissionExplanatoryTextBlock",
    "DetailsOfWasteManagementPracticesAdoptedInYourEstablishmentsAndTheStrategyAdoptedByCompanyToReduceUsageOfHazardousAndToxicChemicalsExplanatoryTextBlock",
    "DisclosureWebLinkOfEntityAtWhichBusinessContinuityAndDisasterManagementPlanIsPlaced",

    # P8 — Inclusive Growth
    "DescribeTheMechanismsToReceiveAndRedressGrievancesOfTheCommunityExplanatoryTextBlock",
    "DetailsOfMeasuresUndertakenByTheEntityToEnsureThatStatutoryDuesHaveBeenDeductedAndDepositedByTheValueChainPartnersExplanatoryTextBlock",
    "DetailsOfAnyCorrectiveActionTakenOrUnderwayToAddressSafetyRelatedIncidentsOnAssessmentOfValueChainPartnersExplanatoryTextBlock",

    # P9 — Consumer Responsibility
    "DescribeTheMechanismsInPlaceToReceiveAndRespondToConsumerComplaintsAndFeedbackExplanatoryTextBlock",
    "DetailsOfImpactOfDataBreaches",
    "DetailsOfAnyCorrectiveActionsTakenOrUnderwayOnIssuesRelatingToAdvertisingAndDeliveryOfEssentialServicesOrCyberSecurityAndDataPrivacyOrRecallsOrPenaltyOrActionTakenByRegulatoryAuthoritiesOnSafetyOfProductsOrServicesExplanatoryTextBlock",
}

# Build a lowercase → original mapping so we can match case-insensitively
# but still store the canonical CamelCase name in the master sheet.
LOWER_TO_CANONICAL = {e.lower(): e for e in RELEVANT_ELEMENTS}

In [3]:
# ---------------------------------------------------------------------------
# HELPER FUNCTIONS
# ---------------------------------------------------------------------------

def parse_filename(filename):
    """Extract company name and year from filename like 'Company Name_2023_24.xlsx'."""
    name = filename.replace(".xlsx", "")
    parts = name.split("_")
    if len(parts) >= 3:
        year = parts[-2] + "-" + parts[-1]
        company = " ".join(parts[:-2])
    elif len(parts) == 2:
        year = parts[-1]
        company = parts[0]
    else:
        year = "Unknown"
        company = name
    return company.title(), year


def is_meaningful_text(x):
    """Return True only if the cell contains a substantive text response."""
    if pd.isna(x):
        return False
    text = str(x).strip()
    if len(text) <= 10:
        return False
    if text.lower() in {"na", "n/a", "-", "none", "0", "not applicable", "nil"}:
        return False
    # Reject cells that are purely numeric
    try:
        float(text.replace(",", ""))
        return False
    except ValueError:
        pass
    return True


def clean_text(text):
    """Collapse whitespace and strip."""
    return re.sub(r"\s+", " ", str(text)).strip()


def find_column(columns, candidates):
    """
    Case-insensitive search for the first matching column name.
    `candidates` is a list of possible names (lowercase).
    Returns the actual column name found, or None.
    """
    col_map = {c.strip().lower(): c for c in columns}
    for cand in candidates:
        if cand in col_map:
            return col_map[cand]
    return None

In [4]:
# ---------------------------------------------------------------------------
# MAIN PROCESSING LOOP
# ---------------------------------------------------------------------------
DATASET_PATH = "Excel Files"

master_data = []

# Quality-tracking lists
files_failed    = []   # unreadable / exception
schema_issues   = []   # missing required columns
null_issues     = []   # nulls in critical fields
period_issues   = []   # multiple period values
no_match_files  = []   # readable but 0 matching rows

all_files = [f for f in os.listdir(DATASET_PATH) if f.endswith(".xlsx")]
print(f"Found {len(all_files)} .xlsx files in '{DATASET_PATH}'")

for file in tqdm(all_files):
    filepath = os.path.join(DATASET_PATH, file)

    try:
        df = pd.read_excel(filepath, dtype=str)   # read everything as string to avoid type coercion

        # ---------------------------------------------------------------
        # 1. SCHEMA VERIFICATION
        #    Accept common spelling / spacing variants for the two key cols
        # ---------------------------------------------------------------
        elem_col   = find_column(df.columns, ["element name", "elementname", "element_name"])
        fact_col   = find_column(df.columns, ["fact value", "factvalue", "fact_value"])
        period_col = find_column(df.columns, ["period", "period of report", "reporting period"])

        if elem_col is None or fact_col is None:
            schema_issues.append(file)
            continue

        df.rename(columns={elem_col: "element_name", fact_col: "fact_value"}, inplace=True)
        if period_col and period_col not in ("element_name", "fact_value"):
            df.rename(columns={period_col: "period"}, inplace=True)

        # ---------------------------------------------------------------
        # 2. NULL / MISSING VALUE DETECTION
        # ---------------------------------------------------------------
        null_elem = df["element_name"].isnull().sum()
        null_fact = df["fact_value"].isnull().sum()
        if null_elem > 0 or null_fact > 0:
            null_issues.append((file, {"element_name_nulls": int(null_elem),
                                        "fact_value_nulls":  int(null_fact)}))

        # ---------------------------------------------------------------
        # 3. PERIOD CONSISTENCY CHECK
        # ---------------------------------------------------------------
        if "period" in df.columns:
            unique_periods = df["period"].dropna().unique()
            if len(unique_periods) > 1:
                period_issues.append((file, list(unique_periods)))

        # ---------------------------------------------------------------
        # 4. CASE-INSENSITIVE ELEMENT MATCHING  ← THE KEY FIX
        #    Strip whitespace, compare lowercase, restore canonical name.
        # ---------------------------------------------------------------
        df["element_name_stripped"] = df["element_name"].astype(str).str.strip()
        df["element_lower"]         = df["element_name_stripped"].str.lower()

        df["canonical_element"] = df["element_lower"].map(LOWER_TO_CANONICAL)
        df = df[df["canonical_element"].notna()].copy()

        # ---------------------------------------------------------------
        # 5. FILTER TO MEANINGFUL TEXT
        # ---------------------------------------------------------------
        df = df[df["fact_value"].apply(is_meaningful_text)].copy()

        if df.empty:
            no_match_files.append(file)
            continue

        company, year = parse_filename(file)

        for _, row in df.iterrows():
            master_data.append({
                "company_name" : company,
                "year"         : year,
                "element_name" : row["canonical_element"],   # always CamelCase
                "fact_value"   : clean_text(row["fact_value"])
            })

    except Exception as e:
        files_failed.append((file, str(e)))

print(f"\nProcessing complete. Rows collected: {len(master_data)}")

Found 2480 .xlsx files in 'Excel Files'


100%|██████████| 2480/2480 [02:34<00:00, 16.01it/s]


Processing complete. Rows collected: 195242


In [5]:
# ---------------------------------------------------------------------------
# BUILD & SAVE MASTER SHEET
# ---------------------------------------------------------------------------
master_df = pd.DataFrame(master_data, columns=["company_name", "year", "element_name", "fact_value"])

before_dedup = len(master_df)
master_df.drop_duplicates(inplace=True)
after_dedup  = len(master_df)

master_df.to_excel("Master Sheet.xlsx", index=False)

print("Master Sheet created successfully!")
print(f"  Rows before dedup : {before_dedup}")
print(f"  Rows after  dedup : {after_dedup}")
print(f"  Duplicate rows removed : {before_dedup - after_dedup}")

Master Sheet created successfully!
  Rows before dedup : 195242
  Rows after  dedup : 153015
  Duplicate rows removed : 42227


In [6]:
# ---------------------------------------------------------------------------
# DATA QUALITY REPORT
# ---------------------------------------------------------------------------
print("\n" + "="*50)
print("         DATA QUALITY REPORT")
print("="*50)

print(f"\nTotal files processed     : {len(all_files)}")
print(f"Files failed (unreadable) : {len(files_failed)}")
print(f"Schema issues             : {len(schema_issues)}")
print(f"Files with null fields    : {len(null_issues)}")
print(f"Period inconsistencies    : {len(period_issues)}")
print(f"Files with 0 matches      : {len(no_match_files)}")
print(f"Final rows in Master Sheet: {after_dedup}")

if files_failed:
    print("\n--- Failed Files ---")
    for fname, err in files_failed:
        print(f"  {fname}: {err}")

if schema_issues:
    print("\n--- Schema Issues (missing required columns) ---")
    for fname in schema_issues:
        print(f"  {fname}")

if null_issues:
    print("\n--- Null Issues ---")
    for fname, counts in null_issues:
        print(f"  {fname}: {counts}")

if period_issues:
    print("\n--- Period Inconsistencies ---")
    for fname, periods in period_issues:
        print(f"  {fname}: {periods}")


         DATA QUALITY REPORT

Total files processed     : 2480
Files failed (unreadable) : 0
Schema issues             : 0
Files with null fields    : 2387
Period inconsistencies    : 2480
Files with 0 matches      : 0
Final rows in Master Sheet: 153015

--- Null Issues ---
  jindal_stainless_ltd_2022_2023.xlsx: {'element_name_nulls': 0, 'fact_value_nulls': 2}
  sanghvi_movers_ltd_2023_2024.xlsx: {'element_name_nulls': 0, 'fact_value_nulls': 19}
  texmaco_rail_engineering_ltd_2023_2024.xlsx: {'element_name_nulls': 0, 'fact_value_nulls': 33}
  sbfc_finance_ltd_2023_2024.xlsx: {'element_name_nulls': 0, 'fact_value_nulls': 17}
  idbi_bank_ltd_2023_2024.xlsx: {'element_name_nulls': 0, 'fact_value_nulls': 27}
  omaxe_ltd_2023_2024.xlsx: {'element_name_nulls': 0, 'fact_value_nulls': 30}
  bajaj_finserv_ltd_2021_2022.xlsx: {'element_name_nulls': 0, 'fact_value_nulls': 25}
  adani_total_gas_ltd_2023_2024.xlsx: {'element_name_nulls': 0, 'fact_value_nulls': 33}
  vadilal_industries_ltd_2023_202

In [7]:
# ---------------------------------------------------------------------------
# QUICK SANITY CHECK — preview a sample of the master sheet
# ---------------------------------------------------------------------------
if not master_df.empty:
    print("\nSample rows from Master Sheet:")
    display(master_df.sample(min(5, len(master_df))))

    print("\nElement distribution (top 20):")
    display(master_df["element_name"].value_counts().head(20).to_frame("count"))

    print("\nCompanies found:", master_df["company_name"].nunique())
    print("Years found:", sorted(master_df["year"].unique()))
else:
    print("Master sheet is empty — check the quality report above for clues.")


Sample rows from Master Sheet:


,company_name,year,element_name,fact_value
72937,Global Health Ltd,2022-2023,DescribeTheProcessesInPlaceToSafelyReclaimYour...,All Hazardous Waste Material is handled as per...
98150,Senco Gold Ltd,2023-2024,PurposeAndScopeOfEngagementIncludingKeyTopicsA...,To understand consumer behaviours and feedback...
44515,Venus Pipes Tubes Ltd,2023-2024,DetailsOfAnyCorrectiveActionTakenOrUnderwayToA...,"All safety incidents, accidents and observatio..."
28844,Castrol India Ltd,2022-2023,InCaseOfRiskApproachToAdaptOrMitigate,CIL adopts an approach of implementing pilot p...
46172,Tube Investments Of India Ltd,2023-2024,InCaseOfRiskApproachToAdaptOrMitigate,Rainwater harvesting mechanisms have been cons...



Element distribution (top 20):


,count
element_name,
RationaleForIdentifyingTheRiskOpportunity,18584
MaterialIssueIdentified,17702
PurposeAndScopeOfEngagementIncludingKeyTopicsAndConcernsRaisedDuringSuchEngagement,14906
InCaseOfRiskApproachToAdaptOrMitigate,13456
TopicsOrPrinciplesCoveredUnderTheTrainingAndItsImpact,6633
NameOfTheNationalAndInternationalCodesOrCertificationsOrLabelsOrStandardsAdoptedByYourEntityAndMappedToEachPrincipleExplanatoryTextBlock,4149
SpecificCommitmentsGoalsAndTargetsSetByTheEntityWithDefinedTimelinesExplanatoryTextBlock,3101
PerformanceOfTheEntityAgainstTheSpecificCommitmentsGoalsAndTargetsAlongWithReasonsInCaseTheSameAreNotMetExplanatoryTextBlock,2641
DescribeTheProcessesForIdentifyingKeyStakeholderGroupsOfTheEntityExplanatoryTextBlock,2450



Companies found: 1265
Years found: ['2020-2021', '2021-2022', '2022-2023', '2023-2024', 'brsr2023']
